l# Invisible Watermarking for Large Language Models via Logit Manipulation
**Ali Hasan (ah2434) & Ammar Syed (as4422) — CS 5788, Spring 2026**

This notebook demonstrates our implementation of the Kirchenbauer et al. (2023) watermarking scheme. It covers:
1. Setup & installation
2. Watermark injection via `WatermarkLogitsProcessor`
3. Watermark detection via z-test
4. Evaluation: detectability, robustness, and text quality (perplexity)
5. Results and plots

> **Note:** Run on a GPU (T4 or better on Google Colab). All results shown in cells below were produced on a single A100 / RTX GPU.

## 1. Setup

In [ ]:
# Install dependencies (run once)
!pip install -q transformers datasets accelerate scipy sentencepiece protobuf

# Clone or mount the repo (on Colab, upload the zip and unzip, or clone from Drive)
# If running locally, skip this cell
import subprocess, sys, os
print('Python:', sys.version)
print('Working dir:', os.getcwd())

In [ ]:
import sys
sys.path.insert(0, '.')  # make sure watermark/ and pipeline/ are importable

import torch
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from transformers import AutoTokenizer, AutoModelForCausalLM

from watermark.logits_processor import WatermarkLogitsProcessor
from watermark.detector import WatermarkDetector
from evaluation.metrics import compute_z_scores, compute_tpr_at_fpr, roc_curve_data
from evaluation.robustness import apply_word_substitution, apply_token_insertion_deletion, evaluate_robustness

matplotlib.rcParams['figure.dpi'] = 120
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Load Model and Tokenizer

We use **LLaMA 3.1 8B Instruct** (Meta) as our primary model and **Gemma 2 9B IT** (Google) as our secondary model for cross-architecture generalization. Both are loaded in fp16 to fit on a T4/A100.

In [ ]:
# --- Choose model here ---
MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'   # or 'google/gemma-2-9b-it'
DELTA = 2.0   # logit bias for green list
GAMMA = 0.5   # fraction of vocab in green list
SEED  = 42    # secret key

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
VOCAB_SIZE = len(tokenizer)
print(f'Vocab size: {VOCAB_SIZE:,}')

## 3. Watermark Injection

We implement the Kirchenbauer et al. scheme as a `LogitsProcessor`. At each decoding step:
1. Hash the previous token ID with our secret key → deterministic green/red partition
2. Add bias `δ` to all green-list logit scores before softmax
3. Sample normally — the model sees no other change

In [ ]:
watermark_processor = WatermarkLogitsProcessor(
    vocab_size=VOCAB_SIZE,
    delta=DELTA,
    gamma=GAMMA,
    seed=SEED,
)

def generate(prompt: str, watermark: bool, max_new_tokens: int = 200) -> dict:
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
    )
    if watermark:
        gen_kwargs['logits_processor'] = [watermark_processor]

    with torch.no_grad():
        output_ids = model.generate(**gen_kwargs)

    prompt_len = inputs['input_ids'].shape[1]
    completion_ids = output_ids[0, prompt_len:].tolist()
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True)
    return {'completion': completion_text, 'token_ids': completion_ids, 'watermarked': watermark}

# Quick sanity check
PROMPT = 'Explain how neural networks learn in simple terms:'
wm_sample = generate(PROMPT, watermark=True)
uwm_sample = generate(PROMPT, watermark=False)
print('--- Watermarked ---')
print(wm_sample['completion'][:300])
print('\n--- Unwatermarked ---')
print(uwm_sample['completion'][:300])

The two outputs should look qualitatively similar — the watermark is imperceptible to a human reader.

## 4. Watermark Detection

The detector re-derives the green/red partition for each token and counts green-list hits. Under the null hypothesis (no watermark), ~50% of tokens are green. A one-sided z-test flags a significant excess.

In [ ]:
detector = WatermarkDetector(
    vocab_size=VOCAB_SIZE,
    gamma=GAMMA,
    seed=SEED,
    z_threshold=4.0,  # will be calibrated empirically below
)

r_wm  = detector.score_sequence(wm_sample['token_ids'])
r_uwm = detector.score_sequence(uwm_sample['token_ids'])

print(f'Watermarked:   z={r_wm.z_score:.2f}, green={r_wm.green_fraction:.2%}, detected={r_wm.is_watermarked}')
print(f'Unwatermarked: z={r_uwm.z_score:.2f}, green={r_uwm.green_fraction:.2%}, detected={r_uwm.is_watermarked}')

## 5. Corpus Generation

We generate 450 prompts (150 per dataset: CNN/DailyMail, WritingPrompts, TriviaQA) and produce both watermarked and unwatermarked completions, for 900 total samples.

In [ ]:
from pipeline.generate import CorpusGenerator, load_corpus
from pathlib import Path

CORPUS_PATH = 'results/corpus_llama_d2.jsonl'  # generated by: python scripts/generate_corpus.py

if not Path(CORPUS_PATH).exists():
    # This takes ~2-4 hours on a T4/A100. Run scripts/generate_corpus.py first.
    generator = CorpusGenerator(
        model_name=MODEL_NAME,
        delta=DELTA, gamma=GAMMA, seed=SEED,
        max_new_tokens=200,
    )
    corpus = generator.generate_corpus(n_per_dataset=150, output_path=CORPUS_PATH, resume=True)
else:
    corpus = load_corpus(CORPUS_PATH)

wm_items  = [x for x in corpus if x['watermarked']]
uwm_items = [x for x in corpus if not x['watermarked']]
print(f'Corpus: {len(wm_items)} watermarked, {len(uwm_items)} unwatermarked')

# Token length distribution
wm_lens  = [x['n_tokens'] for x in wm_items]
uwm_lens = [x['n_tokens'] for x in uwm_items]
print(f'Watermarked tokens — mean: {np.mean(wm_lens):.1f}, median: {np.median(wm_lens):.0f}')
print(f'Unwatermarked tokens — mean: {np.mean(uwm_lens):.1f}, median: {np.median(uwm_lens):.0f}')

## 6. Detectability Evaluation

We compute z-scores for every sample, calibrate the detection threshold to 1% FPR on the unwatermarked corpus, then report TPR.

In [ ]:
wm_z, uwm_z = compute_z_scores(corpus, detector, tokenizer)

# Calibrate threshold to hit 1% FPR
calibrated_threshold = detector.calibrate_threshold(uwm_z, target_fpr=0.01)
threshold, tpr, fpr_actual = compute_tpr_at_fpr(wm_z, uwm_z, target_fpr=0.01)

print(f'Calibrated z-threshold: {calibrated_threshold:.3f}')
print(f'TPR @ 1% FPR: {tpr:.1%}')
print(f'Actual FPR: {fpr_actual:.2%}')

In [ ]:
# --- Figure 1: Z-score distributions ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(uwm_z, bins=40, alpha=0.7, label='Unwatermarked', color='steelblue', density=True)
ax.hist(wm_z,  bins=40, alpha=0.7, label='Watermarked',   color='tomato',    density=True)
ax.axvline(calibrated_threshold, color='black', linestyle='--', label=f'Threshold (1% FPR) = {calibrated_threshold:.2f}')
ax.set_xlabel('z-score')
ax.set_ylabel('Density')
ax.set_title('Z-Score Distribution')
ax.legend()

# --- Figure 2: TPR vs sequence length ---
ax2 = axes[1]
length_bins = [(0, 50), (50, 100), (100, 150), (150, 200), (200, 999)]
bin_labels = ['<50', '50-100', '100-150', '150-200', '200+']
tprs_by_len = []
for lo, hi in length_bins:
    subset_wm  = [z for item, z in zip(wm_items, wm_z) if lo <= item['n_tokens'] < hi]
    if subset_wm:
        tpr_bin = sum(z > calibrated_threshold for z in subset_wm) / len(subset_wm)
    else:
        tpr_bin = 0.0
    tprs_by_len.append(tpr_bin)

ax2.bar(bin_labels, tprs_by_len, color='tomato', alpha=0.85)
ax2.axhline(0.95, color='black', linestyle='--', label='95% TPR target')
ax2.set_ylim(0, 1.05)
ax2.set_xlabel('Sequence length (tokens)')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('TPR by Sequence Length @ 1% FPR')
ax2.legend()

plt.tight_layout()
plt.savefig('fig1_detectability.pdf', bbox_inches='tight')
plt.show()

## 7. Robustness Evaluation

We test whether the watermark survives adversarial post-hoc modifications: random word substitutions (5–20%) and token insertion/deletion.

In [ ]:
robustness_results = evaluate_robustness(
    corpus=corpus,
    detector=detector,
    tokenizer=tokenizer,
    substitution_rates=[0.05, 0.10, 0.15, 0.20],
)

print(f'\n{"Experiment":<30} {"Mean z":>10} {"TPR @1%FPR":>12}')
print('-' * 55)
for exp_name, results in robustness_results.items():
    z_vals = [r.z_score for r in results]
    tpr = sum(r.z_score > calibrated_threshold for r in results) / len(results)
    print(f'{exp_name:<30} {np.mean(z_vals):>10.2f} {tpr:>11.1%}')

In [ ]:
# --- Figure 2: Robustness to word substitution ---
sub_rates = [0, 5, 10, 15, 20]
sub_tprs  = []

for rate_int in sub_rates:
    if rate_int == 0:
        key = 'baseline'
    else:
        key = f'word_sub_{rate_int}pct'
    r = robustness_results[key]
    tpr = sum(x.z_score > calibrated_threshold for x in r) / len(r)
    sub_tprs.append(tpr)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sub_rates, sub_tprs, marker='o', color='tomato', linewidth=2)
ax.axhline(0.95, color='black', linestyle='--', alpha=0.5, label='95% target')
ax.set_xlabel('Word Substitution Rate (%)')
ax.set_ylabel('True Positive Rate @ 1% FPR')
ax.set_title('Watermark Robustness to Word Substitution')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig('fig2_robustness.pdf', bbox_inches='tight')
plt.show()

## 8. Text Quality Evaluation (Perplexity)

We measure perplexity under GPT-2 (an independent reference model) to check whether the watermark degrades output quality. Lower perplexity = more natural text.

In [ ]:
from evaluation.metrics import compute_perplexity

# Use a small independent scorer (GPT-2) — not the generation model
wm_texts  = [x['completion'] for x in wm_items[:100]]   # subset for speed
uwm_texts = [x['completion'] for x in uwm_items[:100]]

print('Computing perplexity (this may take a few minutes)...')
wm_ppl  = compute_perplexity(wm_texts,  model_name='gpt2', device=DEVICE)
uwm_ppl = compute_perplexity(uwm_texts, model_name='gpt2', device=DEVICE)

print(f'Watermarked perplexity:   mean={np.mean(wm_ppl):.2f}, median={np.median(wm_ppl):.2f}')
print(f'Unwatermarked perplexity: mean={np.mean(uwm_ppl):.2f}, median={np.median(uwm_ppl):.2f}')

In [ ]:
# --- Figure 3: Perplexity vs delta (sweep) ---
# Load pre-computed results for different delta values (run generate.py with delta=[0.5, 1, 2, 4, 8])
# Placeholder: show delta=2.0 result vs unwatermarked baseline

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([uwm_ppl, wm_ppl], labels=['Unwatermarked\n(baseline)', f'Watermarked\n(δ={DELTA})'],
           patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Perplexity (GPT-2 scorer)')
ax.set_title('Text Quality: Watermarked vs. Unwatermarked')
plt.tight_layout()
plt.savefig('fig3_perplexity.pdf', bbox_inches='tight')
plt.show()

## 9. Delta Ablation

We sweep the bias parameter δ to understand the detectability–quality tradeoff.

In [ ]:
import os, json as _json

DELTA_SWEEP_PATH = 'results/delta_sweep_llama.json'

if os.path.exists(DELTA_SWEEP_PATH):
    with open(DELTA_SWEEP_PATH) as _f:
        sweep = _json.load(_f)
    deltas = [d['delta'] for d in sweep]
    tprs_d = [d['tpr'] for d in sweep]
    ppls_d = [d['mean_ppl_wm'] for d in sweep]
    print(f'Loaded delta sweep from {DELTA_SWEEP_PATH} ({len(sweep)} points)')
else:
    # Placeholder — replace with real results after running: python scripts/eval_delta_sweep.py
    print(f'WARNING: {DELTA_SWEEP_PATH} not found. Using placeholder values.')
    deltas = [0.5, 1.0, 2.0, 4.0, 8.0]
    tprs_d = [0.62, 0.81, 0.96, 0.99, 1.00]
    ppls_d = [28.1, 29.4, 31.2, 38.7, 61.3]

fig, ax1 = plt.subplots(figsize=(7, 4))
color1, color2 = 'tomato', 'steelblue'
ax1.plot(deltas, tprs_d, 'o-', color=color1, label='TPR @ 1% FPR')
ax1.set_xlabel('Logit bias δ')
ax1.set_ylabel('TPR @ 1% FPR', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(deltas, ppls_d, 's--', color=color2, label='Perplexity (GPT-2)')
ax2.set_ylabel('Perplexity (GPT-2)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax1.set_title('Detectability vs. Quality Trade-off as δ Varies')
plt.tight_layout()
plt.savefig('fig4_delta_ablation.pdf', bbox_inches='tight')
plt.show()

## 10. Results Summary

In [ ]:
print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)
print(f'Model:                    {MODEL_NAME}')
print(f'δ (logit bias):           {DELTA}')
print(f'γ (green list fraction):  {GAMMA}')
print(f'Corpus size:              {len(wm_items)} watermarked, {len(uwm_items)} control')
print()
print(f'Detection (all lengths):')
print(f'  Calibrated z-threshold: {calibrated_threshold:.3f}')
print(f'  TPR @ 1% FPR:           {tpr:.1%}')
print()
print(f'Robustness:')
for exp_name in ['baseline', 'word_sub_10pct', 'word_sub_20pct', 'token_deletion_10pct', 'token_insertion_10pct']:
    if exp_name in robustness_results:
        r = robustness_results[exp_name]
        t = sum(x.z_score > calibrated_threshold for x in r) / len(r)
        print(f'  {exp_name:<35} TPR={t:.1%}')
print()
print(f'Quality (GPT-2 perplexity):')
print(f'  Unwatermarked: {np.mean(uwm_ppl):.2f}')
print(f'  Watermarked:   {np.mean(wm_ppl):.2f}')
print(f'  Relative increase: {(np.mean(wm_ppl)/np.mean(uwm_ppl)-1)*100:.1f}%')